In [1]:
from spaed import spaed
import pandas as pd
import matplotlib.pyplot as plt

import json
import itertools
import numpy as np
from spaed import spaed_, load_pae, get_delineations
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
def load_predictions(filename):
    df = pd.read_csv(filename, index_col= 0)
    df.index = df.index.astype(str)
    df.linkers = df.linkers.astype(str)
    df.disordered = df.disordered.astype(str)
    df.domains = df.domains.astype(str)
    return df

In [3]:
def calc_iou_per_domain(pred_dom, gt_dom):
    inter = len(set(pred_dom) & set(gt_dom))
    union = len(set(pred_dom).union(gt_dom))
    return inter / union

def get_domains(delin):
    domains = []
    for i in delin.split(";"):
        if (i == 'nan') | (i == ""): return []

        start = int(i.split("-")[0])
        end = int(i.split("-")[1])

        reg = list(range(start, end+1))
        domains += [reg]
    return domains

def permut(l1, l2):
    unique = []
    if len(l1) >= len(l2):
        permut = itertools.permutations(l1, len(l2))
        for comb in permut:
            zipped = zip(comb, l2)
            unique.append(list(zipped))
    else:
        permut = itertools.permutations(l2, len(l1))
        for comb in permut:
            zipped = zip(l1, comb)
            unique.append(list(zipped))
    
    return unique

def calc_iou_disc(GTdoms, PREDdoms):
    scores = pd.DataFrame(index=range(len(PREDdoms)), columns=range(len(GTdoms)))

    for i, GTdom in enumerate(GTdoms):
        for j, PREDdom in enumerate(PREDdoms):
            iou_dom = calc_iou_per_domain(PREDdom, GTdom)
            scores.iloc[j, i] = iou_dom

    #Generate dom and GT combinations
    import itertools
    c = permut(scores.index, scores.columns)

    #Find the best matches between preds and GT, such that GT domains correspond only to one pred domain and vice versa
    saved = [0, 0]
    for i in c:
        sum_iou = 0
        for j in i:
            sum_iou += scores.iloc[j]
        if sum_iou > saved[1]:
            saved = [i, sum_iou]

    #calc final iou score
    len_all_doms = len([x for xs in GTdoms for x in xs])
    final=0

    for i in saved[0]:
        final += scores.iloc[i] * ( len(GTdoms[i[1]]) / len_all_doms )
    return final

In [7]:
def fetch_gt_dom(id, gt):
    dom1 = gt.loc[id].dom1.split(",")
    dom2 = gt.loc[id].dom2.split(",")
    if gt.loc[id].dom3 == None:
        dom3 = gt.loc[id].dom3.split(",")
    
    gtdom1 = []; gtdom2=[]; gtdom3=[]
    for j in dom1:
        gtdom1 += get_domains(j)[0]
    for k in dom2:
        gtdom2 += get_domains(k)[0]
    if gt.loc[id].dom3 == None:
        for l in dom3:
            gtdom3 += gt_domains(l)[0]
        gtdoms = [gtdom1, gtdom2, gtdom3]
    else:
        gtdoms = [gtdom1, gtdom2]
    return gtdoms

In [8]:
def fetch_chainsaw_dom_disc(id, chainsaw):
    doms = chainsaw.loc[chainsaw.chain_id == id,"chopping"].values[0].split(",")

    all_doms = []
    for i in doms:
        dom=[]
        for j in i.split("_"):
            dom += get_domains(j)[0]
        all_doms.append(dom)

    return all_doms

In [9]:
def fetch_spaed_dom(id, spaed):
    doms = spaed.loc[id].domains.split(";")

    all_doms = []
    for i in doms:
        all_doms.append(get_domains(i)[0])

    return all_doms

In [4]:
preds = load_predictions("~/Downloads/casp_SPAED.csv")

In [6]:
chainsaw = pd.read_csv("~/Downloads/casp_chainsaw.csv", sep="\t")

In [5]:
gt = pd.read_csv("~/Downloads/casp_gt.csv", index_col=0)

In [13]:
scores = pd.DataFrame(index=gt.index, columns=["SPAED", "Chainsaw"])
for i in gt.index:
    gtdoms = fetch_gt_dom(i, gt)
    SPAEDdoms = get_domains(preds.loc[i].domains)
    chainsawdoms = fetch_chainsaw_dom_disc(i, chainsaw)
    scores.loc[i, "SPAED"] = calc_iou_disc(gtdoms, SPAEDdoms)
    scores.loc[i, "Chainsaw"] = calc_iou_disc(gtdoms, chainsawdoms)

In [30]:
#scores.to_csv("~/Downloads/casp12_eval.csv")

In [14]:
scores

,SPAED,Chainsaw
t0863,0.419869,0.496161
t0890,0.989576,0.895288
t0892,0.736195,0.913257
t0893,0.987603,0.946281
t0894,0.839472,0.819969
t0896,0.675746,0.66971
t0897,0.87892,0.833719
t0898,1.0,0.923077
t0918,0.935658,0.909546
t0920,0.994718,0.544014
